In [23]:
import pandas as pd

# Read the input TSV files
dr_pairs_df = pd.read_csv('data/IN_filtered_DR_100pairs.tsv', sep='\t')
msa_mutant_frequency_df = pd.read_csv('data/IN_MSA_mutant_frequency.tsv', sep='\t')
print (dr_pairs_df.head())
print (msa_mutant_frequency_df.head())

  wildtype_aa1  pos1 mutate_aa1 wildtype_aa2  pos2 mutate_aa2      ddE  \
0            C   140          D            D   148          B  8.50879   
1            D   143          A            D   230          C  6.37754   
2            B   157          A            D   160          B  6.08820   
3            C   140          A            D   148          A  5.61299   
4            C   140          D            D   148          C  5.20420   

   Mutant_MSA_frequency1(based_on_unreduced)  \
0                                   0.199180   
1                                   0.023770   
2                                   0.064754   
3                                   0.011475   
4                                   0.199180   

   Mutant_MSA_frequency2(based_on_unreduced) Unreduced_DRM_with_freq_lt_0.01  
0                                   0.186066                     C140D;D148B  
1                                   0.022951                     D143A;D230C  
2                            

IN_filtered_DR_100pairs.tsv format:
wildtype_aa1	pos1	mutate_aa1	wildtype_aa2	pos2	mutate_aa2	ddE	Mutant_MSA_frequency1(based_on_unreduced)	Mutant_MSA_frequency2(based_on_unreduced)	Unreduced_DRM_with_freq_lt_0.01
C	140	D	D	148	B	8.50879	0.19918032786885245	0.1860655737704918	C140D;D148B
D	143	A	D	230	C	6.37754	0.023770491803278688	0.022950819672131147	D143A;D230C
B	157	A	D	160	B	6.0882	0.06475409836065574	0.0	B157A;

IN_MSA_mutant_frequency.tsv format: 
Position	Consensus	DRM	Reduced_Consensus	Reduced_DRM	Mutant_MSA_frequency(based_on_unreduced)	Mutant_MSA_frequency(based_on_reduced)
140	G	S	C	D	0.19918032786885245	0.19918032786885245
155	N	H	A	C	0.19836065573770492	0.19836065573770492
148	Q	H	D	B	0.1860655737704918	0.1860655737704918
151	V	I	A	B	0.09836065573770492	0.09836065573770492
97	T	A	A	B	0.09098360655737706	0.09098360655737706
157	E	Q	B	A	0.06475409836065574	0.06475409836065574

match the freq df to dr pairs df based on unreduced sequence consensus,pos, mutant, and frequency tuple, when matched, append the unreduced pairs to dr pairs df inthe format of mutation pair (unreduced)"G140S-Q148H" mutation pair (reduced)"C140D-D148B"

In [24]:
# Define a function to create mutation pair strings
def create_mutation_pair(row, freq_df):
    # Match the first mutation
    match1 = freq_df[
        (freq_df['Reduced_Consensus'] == row['wildtype_aa1']) &
        (freq_df['Position'] == row['pos1']) &
        (freq_df['Reduced_DRM'] == row['mutate_aa1']) &
        (freq_df['Mutant_MSA_frequency(based_on_unreduced)'] == row['Mutant_MSA_frequency1(based_on_unreduced)'])
    ]

    # Match the second mutation
    match2 = freq_df[
        (freq_df['Reduced_Consensus'] == row['wildtype_aa2']) &
        (freq_df['Position'] == row['pos2']) &
        (freq_df['Reduced_DRM'] == row['mutate_aa2']) &
        (freq_df['Mutant_MSA_frequency(based_on_unreduced)'] == row['Mutant_MSA_frequency2(based_on_unreduced)'])
    ]

    # Extract unreduced and reduced mutation strings
    mutation1_unreduced = f"{match1['Consensus'].values[0]}{row['pos1']}{match1['DRM'].values[0]}" if not match1.empty else None
    mutation2_unreduced = f"{match2['Consensus'].values[0]}{row['pos2']}{match2['DRM'].values[0]}" if not match2.empty else None
    mutation1_reduced = f"{row['wildtype_aa1']}{row['pos1']}{row['mutate_aa1']}"
    mutation2_reduced = f"{row['wildtype_aa2']}{row['pos2']}{row['mutate_aa2']}"

    # Combine the results into strings
    mutation_pair_unreduced = f"{mutation1_unreduced}-{mutation2_unreduced}" if mutation1_unreduced or mutation2_unreduced else None
    mutation_pair_reduced = f"{mutation1_reduced}-{mutation2_reduced}" if mutation1_reduced or mutation2_reduced else None

    # Extract Mutant_MSA_frequency(based_on_reduced) values
    freq1_reduced = match1['Mutant_MSA_frequency(based_on_reduced)'].values[0] if not match1.empty else None
    freq2_reduced = match2['Mutant_MSA_frequency(based_on_reduced)'].values[0] if not match2.empty else None

    return pd.Series([mutation_pair_unreduced, mutation_pair_reduced, freq1_reduced, freq2_reduced])

# Apply the function to dr_pairs_df
dr_pairs_df[['mutation_pair_unreduced', 'mutation_pair_reduced', 
             'Mutant_MSA_frequency1(based_on_reduced)', 
             'Mutant_MSA_frequency2(based_on_reduced)']] = dr_pairs_df.apply(create_mutation_pair, axis=1, freq_df=msa_mutant_frequency_df)

# Print the updated dataframe
print(dr_pairs_df.head())

  wildtype_aa1  pos1 mutate_aa1 wildtype_aa2  pos2 mutate_aa2      ddE  \
0            C   140          D            D   148          B  8.50879   
1            D   143          A            D   230          C  6.37754   
2            B   157          A            D   160          B  6.08820   
3            C   140          A            D   148          A  5.61299   
4            C   140          D            D   148          C  5.20420   

   Mutant_MSA_frequency1(based_on_unreduced)  \
0                                   0.199180   
1                                   0.023770   
2                                   0.064754   
3                                   0.011475   
4                                   0.199180   

   Mutant_MSA_frequency2(based_on_unreduced) Unreduced_DRM_with_freq_lt_0.01  \
0                                   0.186066                     C140D;D148B   
1                                   0.022951                     D143A;D230C   
2                         

now with the df above, generate a new tsv with follwing columns: MutPair (unreduced), MutPair(reduced), FreqMut1(unreduced;reduced), FreqMut2(unreduced;reduced), and empty column of "StanfordStatus", "ReasonSelection"

In [25]:
# Create a new DataFrame with the required columns
output_df = dr_pairs_df[['mutation_pair_unreduced', 'mutation_pair_reduced']].copy()
output_df.rename(columns={
    'mutation_pair_unreduced': 'MutPair (unreduced)',
    'mutation_pair_reduced': 'MutPair (reduced)'
}, inplace=True)

# Add frequency columns
output_df['FreqMut1 (unreduced;reduced)'] = dr_pairs_df.apply(
    lambda row: f"{row['Mutant_MSA_frequency1(based_on_unreduced)']};{row['Mutant_MSA_frequency1(based_on_reduced)']}", axis=1
)
output_df['FreqMut2 (unreduced;reduced)'] = dr_pairs_df.apply(
    lambda row: f"{row['Mutant_MSA_frequency2(based_on_unreduced)']};{row['Mutant_MSA_frequency2(based_on_reduced)']}", axis=1
)

# Add empty columns
output_df['StanfordStatus'] = ''
output_df['ReasonSelection'] = ''

# Save the DataFrame to a TSV file
output_df.to_csv('output.tsv', sep='\t', index=False)

# Print the resulting DataFrame
print(output_df)

   MutPair (unreduced) MutPair (reduced)  \
0          G140S-Q148H       C140D-D148B   
1          Y143C-S230R       D143A-D230C   
2           E157Q-None       B157A-D160B   
3          G140A-Q148K       C140A-D148A   
4          G140S-Q148R       C140D-D148C   
5          G140S-Q148K       C140D-D148A   
6          G140A-Q148R       C140A-D148C   
7          E138K-Q148K       D138A-D148A   
8          G140A-Q148H       C140A-D148B   
9           E157Q-None       B157A-D160A   
10         Y143A-S230R       D143B-D230C   
11           None-L74M         D72C-A74B   
12         E138K-Q148R       D138A-D148C   
13         G140C-S147G       C140B-B147A   
14          None-G163K       C156A-A163C   
15         G140S-Y143R       C140D-D143C   
16          None-G163K       D119B-A163C   
17          None-Q148R        D39A-D148C   
18         Q148R-N155H       D148C-A155C   
19         Y143R-Q148H       D143C-D148B   
20          None-D232N       C122B-C232D   
21         G140S-S147G       C14